# 01 — Exploratory data analysis: IEEE-CIS

Purpose: answer the questions that shape later phases, not to describe every column.

| Question | Feeds |
|---|---|
| How stable is the fraud rate over time? | threshold selection, prior-shift monitoring (PLAN §5.6, §6) |
| How much do features drift *naturally*, with no injected drift? | drift thresholds and false-alarm budget (§6.3) |
| Do raw `D*` columns drift by construction, and does normalizing them fix it? | feature engineering (§5.3) |
| How is missingness structured? | V-column reduction, identity-optional API contract (§5.3) |
| Which interpretable columns carry signal? | baseline features (§5.2), drift scenarios (§6.5) |

Run from the repo root with `make eda`. Figures go to `reports/figures/eda/`; key numbers to
`reports/eda_summary.json`. Only aggregates are shown — no raw rows (dataset licence).
Outputs are stripped before commit; the write-up with figures is `docs/eda.md`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fraudwatch.config import load_data_config
from fraudwatch.data import columns as c
from fraudwatch.data.manifest import Manifest

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_data_config(ROOT / "configs" / "data.yaml")
PROCESSED = ROOT / cfg.processed_dir
FIG_DIR = ROOT / "reports" / "figures" / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)
manifest = Manifest.read(PROCESSED)
SPLITS = list(manifest.splits)  # train, validation, test, stream (gap is not on disk)
summary: dict = {}
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
# Chart style: reference palette (dataviz skill), light surface, recessive grid, thin marks.
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE = (
    "#2a78d6",
    "#eb6834",
)  # slot 1 = legit / primary series, slot 2 = fraud / second series
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 12,
        "axes.titleweight": "semibold",
        "axes.titlelocation": "left",
        "axes.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": INK2,
        "ytick.color": INK2,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "lines.linewidth": 2,
        "legend.frameon": False,
        "legend.fontsize": 9,
        "font.size": 10,
    }
)


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=160, bbox_inches="tight")


def shade_splits(ax, label_y=None):
    """Thin boundary lines + labels for the time splits; the gap is hatched."""
    for name, r in cfg.splits.items():
        if name == "gap":
            ax.axvspan(r.start_day, r.end_day, color=GRID, alpha=0.6, lw=0)
        if r.start_day > 0:
            ax.axvline(r.start_day, color=INK2, lw=0.8, ls=":")
        if label_y is not None:
            end = r.end_day if r.end_day is not None else 182
            ax.text(
                (r.start_day + end) / 2,
                label_y,
                name,
                ha="center",
                va="bottom",
                color=INK2,
                fontsize=8,
                transform=ax.get_xaxis_transform(),
            )


def load(columns=None):
    frames = []
    for name in SPLITS:
        df = pd.read_parquet(PROCESSED / manifest.splits[name].path, columns=columns)
        frames.append(df.assign(split=name))
    out = pd.concat(frames, ignore_index=True)
    out["split"] = pd.Categorical(out["split"], categories=SPLITS, ordered=True)
    return out

## 1. Splits at a glance (from the data manifest)

In [ ]:
overview = pd.DataFrame(
    {
        name: {
            "rows": s.rows,
            "frauds": s.frauds,
            "fraud_rate": s.fraud_rate,
            "first_day": s.min_day,
            "last_day": s.max_day,
        }
        for name, s in manifest.splits.items()
    }
).T
overview.loc["gap (dropped)", "rows"] = manifest.dropped_rows["gap"]
summary["splits"] = {
    k: {"rows": int(v.rows), "fraud_rate": round(v.fraud_rate, 4)}
    for k, v in manifest.splits.items()
}
overview

## 2. Volume and fraud rate over time

Two panels on a shared time axis (not a dual-axis chart).

In [ ]:
base = load([c.TIME_COL, c.EVENT_DAY_COL, c.TARGET_COL, c.AMOUNT_COL, "ProductCD"])
base["day"] = np.floor(base[c.EVENT_DAY_COL]).astype(int)
base["week"] = base["day"] // 7
gap = cfg.splits["gap"]
all_days = range(0, int(base["day"].max()) + 1)
daily = base.groupby("day").size().reindex(all_days)  # NaN on gap days -> line breaks there
weekly = base.groupby("week").agg(rows=(c.TARGET_COL, "size"), fraud_rate=(c.TARGET_COL, "mean"))
weekly = weekly.reindex(range(0, int(base["week"].max()) + 1))
touches_gap = (weekly.index * 7 < gap.end_day) & (weekly.index * 7 + 7 > gap.start_day)
weekly.loc[touches_gap | (weekly["rows"] < 1000), "fraud_rate"] = np.nan  # partial weeks
weekly_mid_day = weekly.index * 7 + 3.5

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, height_ratios=[1, 1.2])
ax1.plot(daily.index, daily.to_numpy(), color=BLUE, lw=1.5)
ax1.set_title("Transactions per day")
ax1.set_ylim(bottom=0)
shade_splits(ax1, label_y=1.02)
ax2.plot(weekly_mid_day, weekly["fraud_rate"] * 100, color=ORANGE, marker="o", ms=4)
ax2.set_title("Weekly fraud rate (%)")
ax2.set_xlabel("Days since start of data")
ax2.set_ylim(bottom=0)
shade_splits(ax2)
fig.tight_layout()
save(fig, "01_volume_and_fraud_rate")

summary["weekly_fraud_rate"] = {
    "min": round(float(weekly["fraud_rate"].min()), 4),
    "max": round(float(weekly["fraud_rate"].max()), 4),
    "std": round(float(weekly["fraud_rate"].std()), 4),
}
summary["daily_volume"] = {
    "min": int(daily.min()),
    "median": int(daily.median()),
    "max": int(daily.max()),
}
summary["weekly_fraud_rate"]["n_full_weeks"] = int(weekly["fraud_rate"].notna().sum())
summary["weekly_fraud_rate"], summary["daily_volume"]

## 3. Product mix: fraud rate by product, and how the mix shifts across splits

In [ ]:
by_product = base.groupby("ProductCD", observed=True)[c.TARGET_COL].agg(["size", "mean"])
by_product.columns = ["rows", "fraud_rate"]
by_product = by_product.sort_values("fraud_rate", ascending=False)
share = pd.crosstab(base["split"], base["ProductCD"], normalize="index").T.loc[by_product.index]

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.barh(by_product.index, by_product["fraud_rate"] * 100, color=BLUE, height=0.6)
ax.bar_label(
    bars, labels=[f"{v:.1%}" for v in by_product["fraud_rate"]], padding=3, color=INK, fontsize=9
)
ax.invert_yaxis()
ax.set_title("Fraud rate by ProductCD (all splits)")
ax.set_xlabel("Fraud rate (%)")
ax.grid(axis="y", visible=False)
fig.tight_layout()
save(fig, "02_fraud_rate_by_product")

summary["product_fraud_rate"] = by_product["fraud_rate"].round(4).to_dict()
summary["product_share_by_split"] = share.round(4).to_dict()
display(by_product)
share.style.format("{:.1%}").set_caption("Share of transactions by product, per split")

## 4. Transaction amount

In [ ]:
amt = base[c.AMOUNT_COL]
log_bins = np.logspace(np.log10(max(amt.min(), 0.1)), np.log10(amt.max()), 60)
fig, ax = plt.subplots(figsize=(8, 3.5))
for label, color, mask in [
    ("Legit", BLUE, base[c.TARGET_COL] == 0),
    ("Fraud", ORANGE, base[c.TARGET_COL] == 1),
]:
    ax.hist(amt[mask], bins=log_bins, density=True, histtype="step", color=color, lw=2, label=label)
ax.set_xscale("log")
ax.set_title("TransactionAmt distribution, legit vs fraud (density, log x)")
ax.text(
    0.0,
    -0.28,
    "Density per class on log bins: tiny tails look large. Amounts < $1 are 0.03% of rows.",
    transform=ax.transAxes,
    color=INK2,
    fontsize=8,
)
ax.set_xlabel("Amount (log scale)")
ax.set_yticks([])
ax.legend(loc="upper right")
fig.tight_layout()
save(fig, "03_amount_distribution")

cents = (amt * 100).round() % 100 != 0
amount_stats = base.groupby(c.TARGET_COL)[c.AMOUNT_COL].describe(percentiles=[0.5, 0.9, 0.99])
small = amt < 1
summary["amount_under_1"] = {
    "share_of_rows": round(float(small.mean()), 5),
    "share_of_fraud": round(
        float(base.loc[small, c.TARGET_COL].sum() / base[c.TARGET_COL].sum()), 5
    ),
}
summary["amount_median"] = {
    "legit": round(float(amount_stats.loc[0, "50%"]), 2),
    "fraud": round(float(amount_stats.loc[1, "50%"]), 2),
}
summary["fraud_rate_by_has_cents"] = (
    base.groupby(cents)[c.TARGET_COL]
    .mean()
    .round(4)
    .rename({True: "has_cents", False: "whole_amount"})
    .to_dict()
)
display(amount_stats)
summary["fraud_rate_by_has_cents"]

## 5. Missingness structure

Computed per split (one split in memory at a time).

In [ ]:
groups = {
    "core (amount, product, card)": [c.AMOUNT_COL, "ProductCD", *c.CARD_COLS],
    "address/distance": ["addr1", "addr2", "dist1", "dist2"],
    "email": ["P_emaildomain", "R_emaildomain"],
    "C (counts)": c.C_COLS,
    "D (timedeltas)": c.D_COLS,
    "M (match flags)": c.M_COLS,
    "V (Vesta features)": c.V_COLS,
    "identity (id_*, device)": c.IDENTITY_COLS[1:],
}
miss_rows, train_nan_rate = {}, None
for name in SPLITS:
    df = pd.read_parquet(PROCESSED / manifest.splits[name].path)
    rates = df.isna().mean()
    miss_rows[name] = {g: rates[cols].mean() for g, cols in groups.items()}
    if name == "train":
        train_nan_rate = rates
        has_identity = df["DeviceType"].notna() | df[c.CATEGORICAL_ID_COLS].notna().any(axis=1)
        identity_effect = (
            df.groupby(has_identity)[c.TARGET_COL]
            .agg(["size", "mean"])
            .rename(index={True: "has identity row", False: "no identity row"})
        )
        # V columns sharing an identical missingness mask form natural groups
        # (likely one source system each).
        v_masks = df[c.V_COLS].isna()
        v_pattern_groups = v_masks.T.drop_duplicates().shape[0]
    del df
missing = pd.DataFrame(miss_rows)

summary["missing_rate_by_group"] = missing.round(4).to_dict()
block = ["M7", "V1", "D11"]
block_missing = pd.DataFrame(
    {
        n: pd.read_parquet(PROCESSED / manifest.splits[n].path, columns=block).isna().mean()
        for n in SPLITS
    }
)
summary["missing_shift_block"] = block_missing.round(3).to_dict()
display(
    block_missing.style.format("{:.1%}").set_caption(
        "A block of columns becomes far less missing after train"
    )
)
summary["v_missingness_pattern_groups"] = int(v_pattern_groups)
summary["identity_coverage_train"] = round(
    float(identity_effect.loc["has identity row", "size"] / identity_effect["size"].sum()), 4
)
summary["fraud_rate_by_identity_train"] = identity_effect["mean"].round(4).to_dict()
display(missing.style.format("{:.1%}").set_caption("Mean missing rate by column group"))
display(identity_effect)
print(f"{len(c.V_COLS)} V columns share {v_pattern_groups} distinct missingness masks on train")

## 6. Drift metric used below: PSI

Prototype of the Population Stability Index (the production version is Phase 5's `monitoring/psi.py`):
10 quantile bins from the reference, a separate NaN bin, ε = 1e-4 smoothing. Categorical: top-20
categories + other + NaN. Bands: < 0.10 stable, 0.10–0.25 moderate, ≥ 0.25 significant.

In [ ]:
EPS = 1e-4


def _dist(counts):
    p = counts / counts.sum()
    return np.clip(p, EPS, None)


def psi_numeric(ref: pd.Series, cur: pd.Series, bins: int = 10) -> float:
    ref_valid = ref.dropna()
    if ref_valid.nunique() < 2:
        return float("nan")
    edges = np.unique(np.quantile(ref_valid, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf

    def counts(x):
        return np.append(np.histogram(x.dropna(), edges)[0], x.isna().sum()).astype(float)

    p, q = _dist(counts(ref)), _dist(counts(cur))
    return float(np.sum((q - p) * np.log(q / p)))


def psi_categorical(ref: pd.Series, cur: pd.Series, top_k: int = 20) -> float:
    top = ref.value_counts().index[:top_k]

    def counts(x):
        mapped = x.where(x.isin(top) | x.isna(), "__other__").astype(object).fillna("__nan__")
        return mapped.value_counts()

    cats = [*list(top), "__other__", "__nan__"]
    p = _dist(counts(ref).reindex(cats, fill_value=0).to_numpy(float))
    q = _dist(counts(cur).reindex(cats, fill_value=0).to_numpy(float))
    return float(np.sum((q - p) * np.log(q / p)))


def psi(ref: pd.Series, cur: pd.Series) -> float:
    if pd.api.types.is_numeric_dtype(ref):
        return psi_numeric(ref, cur)
    return psi_categorical(ref, cur)


# Sanity check: no drift between two random halves of train should be ~0.
_rng = np.random.default_rng(0)
_amt = pd.read_parquet(PROCESSED / "train.parquet", columns=[c.AMOUNT_COL])[c.AMOUNT_COL]
_half = _rng.random(len(_amt)) < 0.5
print(f"PSI between random halves of train (expect ~0): {psi(_amt[_half], _amt[~_half]):.5f}")

## 7. D columns: drift by construction, and normalization

`D*` columns are time deltas (e.g. days since the card was first seen). For a returning card they
grow as the calendar moves forward, so their distribution shifts over time *by construction*.
Normalizing to a point in time, `D_norm = day − D`, turns "days since X" into "the day X happened",
which should be stable. This compares PSI(train → stream) for raw vs normalized versions.

In [ ]:
d_cols = [c.EVENT_DAY_COL, *c.D_COLS]
train_d = pd.read_parquet(PROCESSED / "train.parquet", columns=d_cols)
stream_d = pd.read_parquet(PROCESSED / "stream.parquet", columns=d_cols)


def normalize(df, col):
    return np.floor(df[c.EVENT_DAY_COL]) - df[col]


d_psi = pd.DataFrame(
    {
        "raw": {col: psi(train_d[col], stream_d[col]) for col in c.D_COLS},
        "normalized": {
            col: psi(normalize(train_d, col), normalize(stream_d, col)) for col in c.D_COLS
        },
    }
)
d_psi["helps"] = d_psi["normalized"] < d_psi["raw"]

fig, ax = plt.subplots(figsize=(9, 3.6))
x = np.arange(len(d_psi))
w = 0.38
ax.bar(x - w / 2 - 0.01, d_psi["raw"], w, color=BLUE, label="raw D")
ax.bar(x + w / 2 + 0.01, d_psi["normalized"], w, color=ORANGE, label="normalized (day - D)")
for level, text in [(0.10, "0.10 moderate"), (0.25, "0.25 significant")]:
    ax.axhline(level, color=INK2, lw=0.8, ls="--")
    ax.text(
        1.005,
        level,
        text,
        ha="left",
        va="center",
        color=INK2,
        fontsize=8,
        transform=ax.get_yaxis_transform(),
    )
ax.set_xticks(x, d_psi.index)
ax.set_yscale("log")
ax.set_ylim(0.003, 5)
ax.set_title("PSI train → stream for D columns: raw vs normalized", pad=26)
ax.set_ylabel("PSI (log scale)")
ax.grid(axis="x", visible=False)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncols=2)
fig.tight_layout()
save(fig, "04_d_column_drift")

summary["d_column_psi_train_to_stream"] = d_psi[["raw", "normalized"]].round(3).to_dict()
d_psi.sort_values("raw", ascending=False)

## 8. Natural drift across splits (no injected drift)

PSI from train to each later split, for every column. This is the "nothing is wrong" baseline
that the monitoring thresholds must tolerate without alerting.

In [ ]:
all_cols = [
    col
    for col in c.TRANSACTION_COLS + c.IDENTITY_COLS[1:]
    if col not in (c.ID_COL, c.TARGET_COL, c.TIME_COL)
]
train_all = pd.read_parquet(PROCESSED / "train.parquet", columns=all_cols)
drift = {}
for name in ["validation", "test", "stream"]:
    cur = pd.read_parquet(PROCESSED / manifest.splits[name].path, columns=all_cols)
    drift[name] = {col: psi(train_all[col], cur[col]) for col in all_cols}
    del cur
drift = pd.DataFrame(drift)

counts = pd.DataFrame(
    {
        "PSI ≥ 0.10": (drift >= 0.10).sum(),
        "PSI ≥ 0.25": (drift >= 0.25).sum(),
        "of columns": drift.notna().sum(),
    }
)
summary["natural_drift_column_counts"] = counts.astype(int).to_dict()
display(counts)

top = drift["stream"].sort_values(ascending=False).head(15)[::-1]
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.barh(top.index, top.to_numpy(), color=BLUE, height=0.6)
ax.bar_label(bars, labels=[f"{v:.2f}" for v in top], padding=3, color=INK, fontsize=8)
for level in (0.10, 0.25):
    ax.axvline(level, color=INK2, lw=0.8, ls="--")
    ax.text(
        level,
        1.005,
        f"{level:.2f}",
        ha="center",
        va="bottom",
        color=INK2,
        fontsize=8,
        transform=ax.get_xaxis_transform(),
    )
ax.set_title("Most-drifted columns, train → stream (natural drift)", pad=16)
ax.set_xlabel("PSI")
ax.grid(axis="y", visible=False)
fig.tight_layout()
save(fig, "05_natural_drift_top_columns")

interpretable = [
    c.AMOUNT_COL,
    "ProductCD",
    "card4",
    "card6",
    "addr1",
    "dist1",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceType",
    "DeviceInfo",
    "C1",
    "C13",
    "M4",
]
summary["natural_drift_interpretable"] = drift.loc[interpretable].round(3).to_dict()
drift.loc[interpretable]

## 9. Signal in interpretable categoricals

In [ ]:
cat_cols = ["card4", "card6", "P_emaildomain", "DeviceType"]
cats = load([c.TARGET_COL, c.TIME_COL, *cat_cols])


def rate_table(col, min_rows=2000, top=8):
    t = cats.groupby(cats[col].astype(object).fillna("(missing)"))[c.TARGET_COL].agg(
        ["size", "mean"]
    )
    t.columns = ["rows", "fraud_rate"]
    return t[t["rows"] >= min_rows].sort_values("fraud_rate", ascending=False).head(top)


for col in cat_cols:
    display(rate_table(col).style.format({"rows": "{:,}", "fraud_rate": "{:.2%}"}).set_caption(col))
    summary[f"fraud_rate_by_{col}"] = rate_table(col)["fraud_rate"].round(4).to_dict()

# Hour of day: TransactionDT's reference point is unknown, so this is hour *relative* to it.
cats["hour"] = (cats[c.TIME_COL] % 86_400) // 3_600
hourly = cats.groupby("hour")[c.TARGET_COL].mean()
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(hourly.index, hourly.to_numpy() * 100, color=ORANGE, marker="o", ms=4)
ax.set_title("Fraud rate by hour (relative to unknown reference; %)")
ax.set_xlabel("Hour offset")
ax.set_xticks(range(0, 24, 3))
ax.set_ylim(bottom=0)
fig.tight_layout()
save(fig, "06_fraud_rate_by_hour")
summary["hourly_fraud_rate"] = {
    "min": round(float(hourly.min()), 4),
    "max": round(float(hourly.max()), 4),
    "argmax_hour": int(hourly.idxmax()),
}
summary["hourly_fraud_rate"]

In [ ]:
(ROOT / "reports" / "eda_summary.json").write_text(
    json.dumps(summary, indent=2, default=str) + "\n"
)
print("wrote reports/eda_summary.json and", len(list(FIG_DIR.glob("*.png"))), "figures")

## 10. Findings and implications

Numbers below are from `reports/eda_summary.json` as produced by this notebook.

1. **Fraud rate moves a lot week to week** (≈2.1%–5.1% across full weeks); validation sits at 4.05% vs
   3.38% in train. → Thresholds tuned on validation are tuned at a different prior than training;
   prior-shift (alert-rate) monitoring needs tolerance for this natural range before it alerts.
2. **The plan's D-column hypothesis was wrong.** Raw `D*` columns are mostly stable train → stream
   (PSI < 0.1 except D11, D15). "Normalized" `day − D` is effectively a *date*: most stream values lie
   after the end of training, so it drifts by construction (PSI 0.26–2.4). → Use normalized D only as
   part of an entity key (Phase 3 stretch / velocity features), not as a model feature.
3. **A block of columns changes missingness after the training period** (M7–M9, V1–V11, D11:
   ~60–73% missing in train → ~29–42% in validation/test/stream). This is the top source of natural
   drift. Because the drift reference is validation, monitoring won't alert on it, but the model is
   trained on the one period where these columns are mostly empty. → Phase 3 decision: refit the final
   model on train+validation, and/or drop or flag this block.
4. **Browser-version columns drift by nature** (`id_31`: chrome 63 → 66). → Map to browser family before
   modeling and monitoring, or the monitor will alert on every browser release.
5. **Natural drift is concentrated**: ~20 of 430 columns exceed PSI 0.25 vs train in each later split;
   the interpretable core (amount, product, card, email, device) stays below 0.1 vs stream. → Supports
   monitoring a small set of important features with calibrated thresholds (PLAN §6.3).
6. **Strong univariate signal**: ProductCD (C 11.7% vs W 2.0%), identity row present (6.7% vs 2.1%),
   card6 credit vs debit (6.7% vs 2.4%), DeviceType mobile, hour offset (peak ~10% around hour 7 vs
   ~2.3% midday). **Weak**: amount alone (similar medians) and having cents.
7. **Sub-$1 amounts** look prominent in the density plot but are 0.03% of rows and 0.1% of fraud,
   all ProductCD C. Not a meaningful pattern at this scale.
